# Exercise 12: Students and Examinations

**Source:** [LeetCode 1280](https://leetcode.com/problems/students-and-examinations/?envType=study-plan-v2&envId=top-sql-50) (Easy).

Table: `Students`
```
+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| student_id    | int     |
| student_name  | varchar |
+---------------+---------+
```
- `student_id` is the primary key
- and `student_name` is the name of the corresponding student.

Table: `Subjects`
```
+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| subject_name | varchar |
+--------------+---------+
```
- `subject_name` is the primary key.
 

Table: `Examinations`
```
+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| student_id   | int     |
| subject_name | varchar |
+--------------+---------+
```
- no primary key,
- `student_id` is from the `Students` table.
- `subject_name` is from the subject name table.
- Each entry is an exam attempt by a student.

**Task.**
Return `student_id`, `student_name`, and for each `subject_name` the number of times `attended_exams` a student attempted the exam.
Order the result by `student_id` and `subject_name`.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_12 WITH (FORCE);
%sql CREATE DATABASE exercise_12;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_12 --alias exercise_12

Connecting and switching to connection 'exercise_12'

#### Fill the database with content

In [7]:
%%sql
DROP TABLE IF EXISTS Students;

CREATE TABLE Students (
    student_id INTEGER PRIMARY KEY,
    student_name VARCHAR NOT NULL
);

INSERT INTO Students (student_id, student_name)
VALUES
    (1, 'Alice'),
    (2, 'Bob'),
    (13, 'John'),
    (6, 'Alex');

DROP TABLE IF EXISTS Subjects;

CREATE TABLE Subjects (
    subject_name VARCHAR PRIMARY KEY
);

INSERT INTO Subjects (subject_name) VALUES ('Math'), ('Physics'), ('Programming');

DROP TABLE IF EXISTS Examinations;

CREATE TABLE Examinations (
    student_id INTEGER NOT NULL,
    subject_name VARCHAR NOT NULL,
    FOREIGN KEY (student_id) REFERENCES Students(student_id),
    FOREIGN KEY (subject_name) REFERENCES Subjects(subject_name)
);

INSERT INTO Examinations (student_id, subject_name)
VALUES
    (1, 'Math'),
    (1, 'Physics'),
    (1, 'Programming'),
    (2, 'Programming'),
    (1, 'Physics'),
    (1, 'Math'),
    (13, 'Math'),
    (13, 'Programming'),
    (13, 'Physics'),
    (2, 'Math'),
    (1, 'Math');

SELECT * FROM Examinations

Running query in 'exercise_12'

4 rows affected.

3 rows affected.

11 rows affected.

11 rows affected.

student_id,subject_name
1,Math
1,Physics
1,Programming
2,Programming
1,Physics
1,Math
13,Math
13,Programming
13,Physics
2,Math


#### Solve the exercise

**Task.**
Return `student_id`, `student_name`, and for each `subject_name` the number of times `attended_exams` a student attempted the exam.
Order the result by `student_id` and `subject_name`.

In [8]:
%%sql
SELECT
    S.student_id, S.student_name, S.subject_name, COUNT(E.subject_name) AS attended_exams
FROM (
    SELECT
        S.student_id, S.student_name, Sub.subject_name
    FROM 
    Students S, Subjects Sub
) S LEFT JOIN Examinations E
ON S.student_id = E.student_id AND S.subject_name = E.subject_name
GROUP BY 1, 2, 3
ORDER BY student_id, subject_name;

Running query in 'exercise_12'

12 rows affected.

student_id,student_name,subject_name,attended_exams
1,Alice,Math,3
1,Alice,Physics,2
1,Alice,Programming,1
2,Bob,Math,1
2,Bob,Physics,0
2,Bob,Programming,1
6,Alex,Math,0
6,Alex,Physics,0
6,Alex,Programming,0
13,John,Math,1


#### Explanation

Use `CROSS JOIN` in order to obtain a table that has for each student (and student id) all the subjects listed.

In [10]:
%%sql
SELECT
    S.student_id, S.student_name, Sub.subject_name
FROM 
Students S, Subjects Sub

Running query in 'exercise_12'

12 rows affected.

student_id,student_name,subject_name
1,Alice,Math
2,Bob,Math
13,John,Math
6,Alex,Math
1,Alice,Physics
2,Bob,Physics
13,John,Physics
6,Alex,Physics
1,Alice,Programming
2,Bob,Programming


- Then do a `LEFT JOIN` with the `Examinations` table on `student_id` and `subject_name`.
- Group by `student_id`, `student_name` and `subject_name` counting over all attempts from `Examinations`.
- Finally respect the order.

#### Close connection

In [11]:
%sql --close exercise_12
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_12;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)